# trapX LLM Advisory — **`advisory_any_bar.py`** (EXTERNAL users)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_any_bar_colab_external.ipynb)

**Run from ANY Google account — no keys, no Drive setup.** This is the external-user edition of the latest `advisory_any_bar.py` tool. Given a **DATE** and **BAR_TIME** it:

1. Downloads that day's folder **read-only** from the owner's shared Google Drive (`gdown`, public — no mount, no login).
2. Gates on the jsonl, rebuilds the advisory input, and fires **one converged LLM call** — routed through the **owner's collector** (you need **no NVIDIA key**).
3. Sends the run's verbose log back to the **owner** for central collection (`external_user_logs/`).

**Local-parity replay.** When the owner has dropped a `pg_snapshot_<date>.db` into the day folder (produced on the trapX box with `_export_pg_day_snapshot.py`), gdown pulls it down automatically and the embedded `advisory_any_bar.py` **auto-detects** it — the resulting log is byte-identical to a local real-Postgres replay (closes the CSV-only gap: run-cumulative floor/ceiling TRAP, option price-symmetry, jerk-family windowed OI). If the snapshot isn't in the day folder yet, the run still succeeds CSV-only (older behaviour) and the log carries an explicit divergence note.

**Not on Colab.** Breeze 1-second futures data. If the requested bar fires `topbottom_formation`, that one touchpoint's 1-sec sustain reads as unavailable in the log — the verdict still runs.

## ⬇ STEP 1 — Enter your NAME or EMAIL (required to run)
Identifies your run in the owner's central log folder. Until it's filled, the run + log steps **skip** (nothing happens).

In [ ]:
#@title ⬇ Your name or email — then Run all { run: "auto" }
EXTERNAL_USER = ""  #@param {type:"string"}
WHO = EXTERNAL_USER.strip()
if WHO:
    print('Hi', WHO + ' — your log will be saved as', WHO + '_<date>_<time>.log')
else:
    print('=' * 66)
    print('  ENTER YOUR NAME OR EMAIL in the box above, then Run all again.')
    print('  Until then, the run + log steps below SKIP — nothing happens.')
    print('=' * 66)

## 1. Install deps
Pins `gdown==6.1.0` (Colab's bundled gdown can't list the >50-item shared folder). No NVIDIA key is needed on your account — the verdict is computed by the owner's collector.

In [ ]:
# gdown==6.1.0: Colab ships an OLD gdown (5.x) whose folder parser can't
# list a >50-item folder; 6.1.0 can (needed to find the day folder).
!pip install -q 'gdown==6.1.0' requests openai anthropic python-dotenv langgraph langgraph-checkpoint-sqlite

import os
try:
    from google.colab import userdata
    for _k in ('NVIDIA_API_KEY', 'ANTHROPIC_API_KEY'):
        try:
            _v = userdata.get(_k)
        except Exception:
            _v = None
        if _v:
            os.environ[_k] = _v.strip()
except Exception:
    pass

print('LLM      : via owner collector (no NVIDIA key needed on this account)')

## 2. Write the embedded `advisory_any_bar.py` + `skills/` + `project/` + wrapper to disk
Decodes the base64 payloads into `/content/` — the latest tool, all skills, the minimal engine `project/` package (so the v5 layer + jerk/signal backbones recompute), and the `run_advisory.py` wrapper that softens `pg_connect()`'s exit + routes the LLM through the owner collector.

In [ ]:
import base64, json, pathlib

SCRIPT_B64  = "..."
SKILLS_B64  = "..."
PROJECT_B64 = "..."
WRAPPER_B64 = "aW1wb3J0IHN5cywgb3MKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUKaW1wb3J0IGFkdmlzb3J5X2FueV9iYXIgYXMgYWFiCgojIDEpIENvbGFiIGhhcyBubyBQb3N0Z3Jlcy4gcGdfY29ubmVjdCgpIHJhaXNlcyBTeXN0ZW1FeGl0IG9uIGEgZmFpbGVkCiMgICAgY29ubmVjdCAob3IgYSBtaXNzaW5nIC0tcGctc25hcHNob3QgZmlsZSksIHdoaWNoIHRoZSBzY3JpcHQncyByZXBsYXkKIyAgICBDU1Ytb25seSBmYWxsYmFjayAoZXhjZXB0IEV4Y2VwdGlvbikgY2Fubm90IGNhdGNoLiBDb252ZXJ0IGl0IHRvIGEKIyAgICBjYXRjaGFibGUgZXJyb3Igc28gdGhlIGludGVuZGVkIGNvbm49Tm9uZSBDU1Ytb25seSBwYXRoIHJ1bnMgd2hlbiBuZWl0aGVyCiMgICAgcmVhbCBQRyBub3IgYSBzbmFwc2hvdCBpcyBhdmFpbGFibGUuIFdoZW4gYWR2aXNvcnkgYXV0by1kZXRlY3RzIGEKIyAgICBwZ19zbmFwc2hvdF9ZWVlZTU1ERC5kYiBpbiB0aGUgZGF5IGZvbGRlciwgcGdfY29ubmVjdCByZXR1cm5zIHRoZQojICAgIHNxbGl0ZSBzaGltIHN1Y2Nlc3NmdWxseSDigJQgdGhpcyB3cmFwcGVyIGlzIGEgbm8tb3AgaW4gdGhhdCBjYXNlLgpfb3JpZ19wZyA9IGFhYi5wZ19jb25uZWN0CmRlZiBfcGdfb3B0aW9uYWwoKToKICAgIHRyeToKICAgICAgICByZXR1cm4gX29yaWdfcGcoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInBvc3RncmVzIHVuYXZhaWxhYmxlIG9uIENvbGFiOyBDU1Ytb25seTogJXMiICUgZSkKYWFiLnBnX2Nvbm5lY3QgPSBfcGdfb3B0aW9uYWwKCiMgMikgRXh0ZXJuYWwgdXNlcnMgaG9sZCBOTyBOVklESUEga2V5LiBXaGVuIFRSQVBYX0xMTV9QUk9YWSBpcyBzZXQsIHJvdXRlIHRoZQojICAgIGNoYXQtY29tcGxldGlvbiB0aHJvdWdoIHRoZSBvd25lcidzIEFwcHMgU2NyaXB0IHByb3h5IChob2xkcyB0aGUga2V5CiMgICAgc2VydmVyLXNpZGUpLiBTYW1lIHBhcmFtcyAodGVtcGVyYXR1cmUgMCwgc2VlZCA0MikgLT4gc2FtZSByZXN1bHQuCiMgICAgTkI6IHRoZSB3cmFwcGVyIGFjY2VwdHMgKGFuZCBmb3J3YXJkcykgRVZFUlkga3dhcmcgdGhlIGN1cnJlbnQgY2FsbF9udmlkaWEKIyAgICBzaWduYXR1cmUgdGFrZXMg4oCUIGByZXRyaWVzYCB3YXMgYWRkZWQgaW4gQ0hBLTI4OCAoYFJFUExBWV9ERUZBVUxUX1JFVFJJRVNgKSwKIyAgICBhbmQgX2ZpbmlzaF9hbmRfbG9nIG5vdyBwYXNzZXMgaXQgZXhwbGljaXRseS4gTWlzc2luZyBpdCBoZXJlIGlzIGEgaGFyZAojICAgIFR5cGVFcnJvciwgbm90IGEgZ3JhY2VmdWwgZGVncmFkYXRpb24uCmRlZiBfY2FsbF9udmlkaWFfdmlhX3Byb3h5KHByb3h5X3VybCwgc3lzdGVtLCB1c2VyLCBtb2RlbCwgdGltZW91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X3Rva2Vucz1Ob25lLCByZXRyaWVzPU5vbmUpOgogICAgaW1wb3J0IHRpbWUKICAgIGltcG9ydCByZXF1ZXN0cwogICAgcGF5bG9hZCA9IHsiYWN0aW9uIjogImxsbSIsICJtb2RlbCI6IG1vZGVsLCAic3lzdGVtIjogc3lzdGVtLCAidXNlciI6IHVzZXIsCiAgICAgICAgICAgICAgICJ0ZW1wZXJhdHVyZSI6IGFhYi5OVklESUFfVEVNUEVSQVRVUkUsICJzZWVkIjogYWFiLk5WSURJQV9TRUVEfQogICAgaWYgbWF4X3Rva2VucyBpcyBub3QgTm9uZToKICAgICAgICBwYXlsb2FkWyJtYXhfdG9rZW5zIl0gPSBtYXhfdG9rZW5zCiAgICAjIFJldHJ5IFBPU1RzIHRvIHRoZSBBcHBzIFNjcmlwdCBwcm94eSBvbiB0cmFuc2llbnQgdXBzdHJlYW0gZmFpbHVyZXMgc28KICAgICMgdGhlIG5vdGVib29rIHJpZGVzIG91dCB0aGUgc2FtZSA0MjkvNTA0IHN0b3JtIHRoZSBkaXJlY3QgbnZpZGlhIGNhbGxlcgogICAgIyBoYW5kbGVzLiBgcmV0cmllc2AgaXMgdGhlIE5VTUJFUiBvZiBleHRyYSBhdHRlbXB0cyBhZnRlciB0aGUgZmlyc3QuCiAgICBhdHRlbXB0cyA9IGludChyZXRyaWVzKSBpZiByZXRyaWVzIGVsc2UgMAogICAgbGFzdF9leGMgPSBOb25lCiAgICBmb3IgaSBpbiByYW5nZShhdHRlbXB0cyArIDEpOgogICAgICAgIHQwID0gZGF0ZXRpbWUubm93KCkKICAgICAgICB0cnk6CiAgICAgICAgICAgIHIgPSByZXF1ZXN0cy5wb3N0KHByb3h5X3VybCwganNvbj1wYXlsb2FkLCB0aW1lb3V0PXRpbWVvdXQpCiAgICAgICAgICAgIGxhdGVuY3lfbXMgPSAoZGF0ZXRpbWUubm93KCkgLSB0MCkudG90YWxfc2Vjb25kcygpICogMTAwMC4wCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGRhdGEgPSByLmpzb24oKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJMTE0gcHJveHkgcmV0dXJuZWQgbm9uLUpTT04gKEhUVFAgJXMpOiAlcyIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAlIChyLnN0YXR1c19jb2RlLCByLnRleHRbOjIwMF0pKQogICAgICAgICAgICBpZiBub3QgZGF0YS5nZXQoIm9rIiwgRmFsc2UpOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJMTE0gcHJveHkgZXJyb3I6ICVzIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICUgZGF0YS5nZXQoImVycm9yIiwgInVua25vd24iKSkKICAgICAgICAgICAgcmV0dXJuIHsidGV4dCI6IGRhdGEuZ2V0KCJ0ZXh0IiwgIiIpLAogICAgICAgICAgICAgICAgICAgICJsYXRlbmN5X21zIjogcm91bmQobGF0ZW5jeV9tcywgMSksCiAgICAgICAgICAgICAgICAgICAgIm1vZGVsIjogbW9kZWwsCiAgICAgICAgICAgICAgICAgICAgInVzYWdlIjogZGF0YS5nZXQoInVzYWdlIiwge30pIG9yIHt9fQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgbGFzdF9leGMgPSBlCiAgICAgICAgICAgIGlmIGkgPCBhdHRlbXB0czoKICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAobWluKDIgKiogaSwgOCkpICAgIyAxcywgMnMsIDRzLCA4cywgOHMg4oCmCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBicmVhawogICAgcmFpc2UgU3lzdGVtRXhpdCgiTExNIHByb3h5IGZhaWxlZCBhZnRlciAlcyBhdHRlbXB0KHMpOiAlcyIKICAgICAgICAgICAgICAgICAgICAgJSAoYXR0ZW1wdHMgKyAxLCBsYXN0X2V4YykpCgpfb3JpZ19udiA9IGFhYi5jYWxsX252aWRpYQpkZWYgX2NhbGxfbnZpZGlhKHN5c3RlbSwgdXNlciwgbW9kZWwsIHRpbWVvdXQsIG1heF90b2tlbnM9Tm9uZSwgcmV0cmllcz1Ob25lKToKICAgIHByb3h5ID0gb3MuZW52aXJvbi5nZXQoIlRSQVBYX0xMTV9QUk9YWSIsICIiKS5zdHJpcCgpCiAgICBpZiBwcm94eToKICAgICAgICByZXR1cm4gX2NhbGxfbnZpZGlhX3ZpYV9wcm94eShwcm94eSwgc3lzdGVtLCB1c2VyLCBtb2RlbCwgdGltZW91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zPW1heF90b2tlbnMsIHJldHJpZXM9cmV0cmllcykKICAgICMgQ2FsbCB0aGUgcmVhbCBgY2FsbF9udmlkaWFgIOKAlCBmb3J3YXJkIG9ubHkga3dhcmdzIGl0IGFjY2VwdHMsIHNvIGEKICAgICMgZnV0dXJlIHNpZ25hdHVyZSBkcmlmdCBoZXJlIGlzIGEgc29mdC1kZWdyYWRlLCBub3QgYSBoYXJkIFR5cGVFcnJvci4KICAgIGt3ID0ge30KICAgIGlmIG1heF90b2tlbnMgaXMgbm90IE5vbmU6CiAgICAgICAga3dbIm1heF90b2tlbnMiXSA9IG1heF90b2tlbnMKICAgIGlmIHJldHJpZXMgaXMgbm90IE5vbmU6CiAgICAgICAga3dbInJldHJpZXMiXSA9IHJldHJpZXMKICAgIHJldHVybiBfb3JpZ19udihzeXN0ZW0sIHVzZXIsIG1vZGVsLCB0aW1lb3V0LCAqKmt3KQphYWIuY2FsbF9udmlkaWEgPSBfY2FsbF9udmlkaWEKCnN5cy5leGl0KGFhYi5tYWluKCkpCg=="

pathlib.Path('advisory_any_bar.py').write_bytes(base64.b64decode(SCRIPT_B64))
pathlib.Path('run_advisory.py').write_bytes(base64.b64decode(WRAPPER_B64))
skills = json.loads(base64.b64decode(SKILLS_B64).decode('utf-8'))
sd = pathlib.Path('skills'); sd.mkdir(exist_ok=True)
for name, text in skills.items():
    (sd / name).write_text(text, encoding='utf-8')
proj = json.loads(base64.b64decode(PROJECT_B64).decode('utf-8'))
for relpath, text in proj.items():
    p = pathlib.Path(relpath); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8')
print('wrote advisory_any_bar.py + run_advisory.py +',
      len(skills), 'skills +', len(proj),
      'project files (engine v5 recompute + PG-snapshot auto-detect enabled)')

## 3. Choose the bar + backend + model
`WHEN` is the bar in `DD-mon, HH:MM` form. `MODEL` defaults to **`z-ai/glm-5.2`** (current live-engine default); pick `meta/llama-3.3-70b-instruct` for the pre-CHA-319 llama baseline. (The owner's collector must allow the chosen model.)

In [ ]:
#@title Parameters { run: "auto" }
WHEN    = "29-Jun, 09:46"  #@param {type:"string"}
BACKEND = "nvidia"         #@param ["nvidia", "anthropic", "auto"]
YEAR    = 2026             #@param {type:"integer"}
# NVIDIA model id. Default z-ai/glm-5.2 (live default). Pick
# meta/llama-3.3-70b-instruct for the pre-CHA-319 llama baseline, or ""
# (= backend default = z-ai/glm-5.2), or type any id.
MODEL   = "z-ai/glm-5.2"   #@param ["z-ai/glm-5.2", "meta/llama-3.3-70b-instruct", ""] {allow-input: true}
# Optional: paste the day sub-folder's Google Drive id to SKIP the slow
# parent-folder listing (gdown lists >600 items across ~70 day folders,
# which can OOM the Colab free tier and disconnect the runtime). The
# id is the token after 'folders/' in the day folder's Drive URL.
DAY_FOLDER_ID = ""           #@param {type:"string"}
print('WHEN   =', WHEN)
print('BACKEND=', BACKEND, '  YEAR=', YEAR,
      '  MODEL=', MODEL or '(default: z-ai/glm-5.2)')
print('DAY_ID =', DAY_FOLDER_ID or '(auto — listing shared parent)')

## 4. Resolve the day (no Drive mount, no NVIDIA key)
Parses the bar and resolves the local download folder. The data itself is fetched read-only by the run step via `gdown` from the owner's public shared folder — your account only ever **reads**. If the owner has dropped a `pg_snapshot_<date>.db` into the shared folder, gdown will pull it too and the tool auto-picks it up for local-parity replay.

In [ ]:
import sys, os, argparse
sys.path.insert(0, '.')
import advisory_any_bar as aab

# Owner's Apps Script endpoint — LLM proxy + log collector.
COLLECTOR_URL = "https://script.google.com/macros/s/AKfycbx8SDCFUaAf_kxKxUOSpOnb0tsbwQ6OS-F72bjer60LQyI1-uoF7edA0XCvzCpBhtLQ/exec"

# Self-heal: rebuild WHO from the STEP-1 param even if Step 1's cell
# wasn't re-run after typing the name (a Colab `run:auto` widget update
# doesn't always fire). Falls back to '' cleanly if nothing was typed.
WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()

req = aab.parse_request(argparse.Namespace(when=WHEN, date=None,
                                           time=None, year=YEAR))
DAY_FOLDER = req.tmp_dir_name        # gdown target (e.g. 'tmp_Jun_29')
print('you  :', WHO or '(enter name in STEP 1)')
print(f'date {req.iso_date}  ->  download {req.mon_dd!r}, verdict via owner LLM')
print('log  ->  owner  external_user_logs/' + (WHO or '<you>')
      + '_<date>_<time>.log')

## 5. Run the advisory (verdict via the owner's LLM)
Runs the tool through the `run_advisory.py` wrapper that (a) softens `pg_connect`'s `SystemExit` so the CSV-only fallback engages when neither real PG nor a snapshot is around, and (b) routes the LLM call through the owner's collector (`TRAPX_LLM_PROXY`). No `--local-dir` → the script `gdown`-downloads the read-only day folder itself, INCLUDING the day's `pg_snapshot_<date>.db` if present (auto-detected by `advisory_any_bar.py`).

**Streaming output.** Progress lines (gdown downloads, `[GATE]`, `[SKILL-COT]` drills, `[LLM] Firing…`, `[LLM] Done`) print live as they happen — the cell no longer looks frozen while advisory runs.

> **Runtime disconnected during `[DRIVE] Listing shared folder`?** That's the Colab free tier OOM-killing the process while gdown enumerates ~700 items across ~70 day folders. Fix: open the target day folder in Drive (e.g. `Jun_29`), copy the id after `folders/` from the URL, paste it into `DAY_FOLDER_ID` in step 3, and re-run. advisory then downloads that folder directly — no parent listing.

> GLM 5.2 is much faster on nvidia than 5.1 was; `meta/llama-3.3-70b` reflects the pre-CHA-319 live baseline. Either way the `[LLM] Firing…` line can sit for tens of seconds — that's the LLM call, not a hang.

In [ ]:
import subprocess, sys, shlex, os

WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()
if not WHO:
    print('SKIPPED — enter your NAME or EMAIL in STEP 1 (top), then Run all again.')
else:
    cmd = [sys.executable, '-u', 'run_advisory.py', WHEN,
           '--skills-dir', 'skills', '--backend', BACKEND, '--year', str(YEAR),
           '--no-live']
    if MODEL.strip():
        cmd += ['--model', MODEL.strip()]
    # If the operator supplied a direct day-folder id, skip advisory's
    # slow parent-folder listing entirely — no gdown listing = no OOM.
    _did = str(globals().get('DAY_FOLDER_ID') or '').strip()
    if _did:
        cmd += ['--gdrive-day-id', _did]
    env = dict(os.environ)
    env['PYTHONUNBUFFERED'] = '1'   # belt+braces vs Python line buffering
    if COLLECTOR_URL.startswith('http'):
        env['TRAPX_LLM_PROXY'] = COLLECTOR_URL   # LLM -> owner's key
    print('$', ' '.join(shlex.quote(c) for c in cmd), '\n', flush=True)
    # Stream stdout+stderr line-by-line so Colab shows progress live
    # (gdown downloads, [GATE], [DATA], [LLM] Firing…, [LLM] Done). Without
    # this, capture_output=True buffers everything until the subprocess
    # exits and the cell looks frozen for the full 1-5 min run.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, env=env,
                            bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    finally:
        rc = proc.wait()
    if rc != 0:
        print('\n[exit code]', rc)

## 6. The verbose log on this Colab VM
Lists the verbose log(s) written locally this session and prints the newest. Look for the `[PG-SNAPSHOT] auto-detected …` line near the top to confirm local-parity mode; if it's absent the run was CSV-only.

In [ ]:
import glob, os
WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()
if not WHO:
    print('SKIPPED — enter your NAME or EMAIL in STEP 1 (top), then Run all again.')
else:
    logs = sorted(glob.glob(f'{DAY_FOLDER}/advisory_{req.yyyymmdd}_*.log'),
                  key=os.path.getmtime)
    if logs:
        print(f'{len(logs)} verbose log(s) on this VM in {DAY_FOLDER}:')
        for p in logs:
            print('   ', os.path.basename(p), f'({os.path.getsize(p)} bytes)')
        print('\n===== newest:', os.path.basename(logs[-1]), '=====\n')
        print(open(logs[-1], encoding='utf-8').read())
    else:
        print('no verbose log in', DAY_FOLDER,
              '— did the run gate out (no pattern at that bar) or error?')

## 7. Send this run's log to the OWNER (central collection)
POSTs the newest log to the owner's collector, which saves it under `external_user_logs/<you>_<date>_<time>.log` in the **owner's** Drive. Your account never writes to Drive.

In [ ]:
import glob, os
WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()
if not WHO:
    print('SKIPPED — enter your NAME or EMAIL in STEP 1 (top), then Run all again.')
else:
    src_logs = sorted(glob.glob(f'{DAY_FOLDER}/advisory_{req.yyyymmdd}_*.log'),
                      key=os.path.getmtime)
    if not src_logs:
        print('no log to send — run the previous cell first.')
    elif not COLLECTOR_URL.startswith('http'):
        print('collector URL not set — your log is on the Colab VM:')
        print('   ', os.path.abspath(src_logs[-1]))
    else:
        newest = src_logs[-1]
        body = {'user': WHO,
                'when': f'{req.yyyymmdd}_' + req.time.replace(':', ''),
                'log': open(newest, encoding='utf-8').read()}
        try:
            import requests
            resp = requests.post(COLLECTOR_URL, json=body, timeout=90).json()
            if resp.get('ok'):
                print('sent your log to the owner as:', resp.get('saved'))
            else:
                print('collector returned an error:', resp.get('error'))
        except Exception as e:
            print('could not reach collector:', type(e).__name__, e)
            print('your log is on the Colab VM:', os.path.abspath(newest))